# Alternative visual-typicality robustness test (MM visual embeddings)
**Beyond Audio: Music Genre Classification — exploratory follow-up**

**Question:** Does the observed relationship between visual typicality and fusion accuracy persist if we define typicality using **MM visual** embeddings rather than the original **CNN visual** embeddings?

**Important scientific boundaries:**
- This is a **new, post-hoc robustness analysis**, not an original pre-registered test or an independent validation dataset.
- Both definitions use **true test genres** to match observations with **training-set true-genre centroids** and to rank observations within true genre. Neither definition is available at inference time for an unknown-label recording.
- MM embeddings were learned within a multimodal system derived from CNN features. They are **not independent visual evidence**.
- The analysis reuses the already-selected audio-only and early-fusion classifier predictions, with **no retraining and no test-set tuning**. It compares associations, **not causal effects**.
- Run results first; do **not** pre-commit to a conclusion or edit the report before inspecting the output.

## Expected inputs
The same Google Drive project tree as your Stage 3 experiment, with:
- `finetuning_experiments/multimodal_artist_disjoint/labels_train.csv`
- `finetuning_experiments/multimodal_artist_disjoint/labels_test.csv`
- `finetuning_experiments/oramas_representation_experiment/labels_train.csv`
- `finetuning_experiments/oramas_representation_experiment/labels_test.csv`
- `finetuning_experiments/oramas_representation_experiment/X_train_mm-visual.npy`
- `finetuning_experiments/oramas_representation_experiment/X_test_mm-visual.npy`
- `finetuning_experiments/reproducible_stage3_2026/stage3_paired_test_predictions_and_typicality.csv`

All output files go to a **new folder**: `finetuning_experiments/mm_visual_typicality_robustness_2026`.

## 1. Connect the project and verify inputs

In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import paired_cosine_distances

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT = Path(os.environ.get("MSDI_PROJECT_DIR", "/content/drive/MyDrive/MSD-I_project"))
EXP = PROJECT / "finetuning_experiments"
CNN_DIR = EXP / "multimodal_artist_disjoint"
MM_DIR = EXP / "oramas_representation_experiment"
STAGE3_DIR = EXP / "reproducible_stage3_2026"
OUTPUT = EXP / "mm_visual_typicality_robustness_2026"

FILES = {
    "CNN training metadata": CNN_DIR / "labels_train.csv",
    "CNN test metadata": CNN_DIR / "labels_test.csv",
    "MM training metadata": MM_DIR / "labels_train.csv",
    "MM test metadata": MM_DIR / "labels_test.csv",
    "MM training visual embeddings": MM_DIR / "X_train_mm-visual.npy",
    "MM test visual embeddings": MM_DIR / "X_test_mm-visual.npy",
    "Original verified Stage 3 paired results": STAGE3_DIR / "stage3_paired_test_predictions_and_typicality.csv",
}
missing = [f"{name}: {path}" for name, path in FILES.items() if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing input files:\n" + "\n".join(missing) +
                            "\nRun the verified Stage 3 notebook first, or check MSDI_PROJECT_DIR.")

print("Project:", PROJECT)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "pandas:", pd.__version__)
print("All input files found. New output folder:", OUTPUT)

## 2. Verify artist-disjoint labels, alignment, and saved predictions

In [ ]:
meta_cols = ["msd_track_id", "msd_artist_id", "genre", "split"]

def read_labels(path, expected_n, expected_split):
    frame = pd.read_csv(path, dtype={"msd_track_id": str, "msd_artist_id": str})
    assert set(meta_cols).issubset(frame.columns), path
    assert len(frame) == expected_n and frame.msd_track_id.is_unique, path
    assert frame[meta_cols].notna().all().all() and (frame["split"] == expected_split).all()
    return frame.reset_index(drop=True)

cnn_train = read_labels(FILES["CNN training metadata"], 8242, "train")
cnn_test = read_labels(FILES["CNN test metadata"], 1792, "test")
mm_train = read_labels(FILES["MM training metadata"], 8242, "train")
mm_test = read_labels(FILES["MM test metadata"], 1792, "test")
for split, a, b in [("train", cnn_train, mm_train), ("test", cnn_test, mm_test)]:
    for col in meta_cols:
        assert a[col].astype(str).equals(b[col].astype(str)), (split, col)
assert set(cnn_train.msd_artist_id).isdisjoint(set(cnn_test.msd_artist_id))
assert set(cnn_train.msd_track_id).isdisjoint(set(cnn_test.msd_track_id))
assert cnn_test.msd_artist_id.nunique() == 975

paired = pd.read_csv(FILES["Original verified Stage 3 paired results"],
                     dtype={"msd_track_id": str, "msd_artist_id": str})
required = ["msd_track_id", "msd_artist_id", "genre", "typicality_quintile",
            "cnn_audio_correct", "cnn_early_fusion_correct", "mm_audio_correct", "mm_fusion_correct"]
assert set(required).issubset(paired.columns), set(required)-set(paired.columns)
assert len(paired) == 1792 and paired.msd_track_id.is_unique
# Verify metadata by IDs rather than relying on row order in Stage 3's saved table.
audit = cnn_test[meta_cols].merge(paired[meta_cols], on="msd_track_id", how="outer",
                                  indicator=True, suffixes=("_labels", "_stage3"), validate="one_to_one")
assert (audit["_merge"] == "both").all() and len(audit) == 1792
for col in ("msd_artist_id", "genre", "split"):
    assert (audit[col + "_labels"] == audit[col + "_stage3"]).all(), col

for col in required[4:]:
    s = paired[col]
    if pd.api.types.is_bool_dtype(s):
        pass
    elif pd.api.types.is_numeric_dtype(s) and set(s.dropna().unique()) <= {0, 1}:
        paired[col] = s.astype(bool)
    else:
        lookup = {"True": True, "False": False, "1": True, "0": False, "true": True, "false": False}
        parsed = s.astype(str).str.strip().map(lookup)
        assert parsed.notna().all(), f"Unexpected correctness values in {col}"
        paired[col] = parsed.astype(bool)
assert paired[required].notna().all().all()
paired["cnn_visual_quintile"] = paired["typicality_quintile"].astype(int)
assert set(paired["cnn_visual_quintile"]) == {1,2,3,4,5}

print("Train/test track counts: 8,242 / 1,792; artists in test: 975")
print("CNN and MM metadata aligned: YES; Stage 3 predictions aligned: YES")

## 3. Reconstruct MM-based true-genre visual typicality

Fit `StandardScaler` **only on training MM visual embeddings**. Compute centroids from training records by training genre. For each test track, measure cosine distance to its **true-genre** centroid and split the rankings **within each true test genre**, matching the original Stage 3 quintile construction.

In [ ]:
X_train = np.load(FILES["MM training visual embeddings"], allow_pickle=False)
X_test = np.load(FILES["MM test visual embeddings"], allow_pickle=False)
assert X_train.shape == (8242,200) and X_test.shape == (1792,200)
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()

scaler = StandardScaler()
Z_train = scaler.fit_transform(X_train)
Z_test = scaler.transform(X_test)
genres = sorted(cnn_train.genre.unique())
assert genres == sorted(cnn_test.genre.unique())
centroids = {g: Z_train[mm_train.genre.to_numpy() == g].mean(axis=0) for g in genres}
matched_centroids = np.stack([centroids[g] for g in mm_test.genre])
mm_dist = paired_cosine_distances(Z_test, matched_centroids)
assert np.isfinite(mm_dist).all()

# Assign group within each true genre in the original MM test row order.
mm_details = mm_test[meta_cols].copy()
mm_details["mm_visual_distance"] = mm_dist
mm_details["mm_visual_quintile"] = (mm_details.groupby("genre")["mm_visual_distance"]
    .transform(lambda x: pd.qcut(x.rank(method="first"), q=5, labels=False)+1)
    .astype(int))
assert set(mm_details.mm_visual_quintile) == {1,2,3,4,5}
assert mm_details.mm_visual_quintile.value_counts().sum() == 1792

paired = paired.merge(mm_details[["msd_track_id","mm_visual_distance","mm_visual_quintile"]],
                      on="msd_track_id", how="left", sort=False, validate="one_to_one")
assert len(paired) == 1792 and paired[["mm_visual_quintile","mm_visual_distance"]].notna().all().all()

print("MM quintile sizes (Q1 most typical; Q5 most atypical):")
print(paired.mm_visual_quintile.value_counts().sort_index().to_string())
print("\nMM genre-conditional quintile counts:")
print(pd.crosstab(paired.genre, paired.mm_visual_quintile).to_string())
print("\nMETHOD NOTE: true genre is required; this analysis is retrospective.")

## 4. Measure grouping similarity and the paired prediction changes

This comparison uses the **same frozen classifier predictions** under two alternative grouping schemes. We do not fit any new classification model.

In [ ]:
assignments = pd.crosstab(paired.cnn_visual_quintile, paired.mm_visual_quintile,
                          rownames=["Original CNN Q"], colnames=["Alternative MM Q"])
exact_agreement = 100 * (paired.cnn_visual_quintile == paired.mm_visual_quintile).mean()
q5_cnn = set(paired.loc[paired.cnn_visual_quintile.eq(5), "msd_track_id"])
q5_mm = set(paired.loc[paired.mm_visual_quintile.eq(5), "msd_track_id"])
q5_intersection = len(q5_cnn & q5_mm)
q5_jaccard = q5_intersection / len(q5_cnn | q5_mm)
print("CNN-group vs MM-group membership:")
print(assignments.to_string())
print(f"\nExact same quintile: {exact_agreement:.2f}%")
print(f"Q5 overlap: {q5_intersection} tracks; Jaccard similarity {q5_jaccard:.3f}")

comparisons = [
    ("CNN", "cnn_audio_correct", "cnn_early_fusion_correct"),
    ("MM", "mm_audio_correct", "mm_fusion_correct"),
]
rows = []
for grouping, col in [("CNN visual typicality", "cnn_visual_quintile"),
                      ("MM visual typicality", "mm_visual_quintile")]:
    for q in range(1,6):
        sub = paired.loc[paired[col].eq(q)]
        for model, audio, fusion in comparisons:
            rescued = int((~sub[audio] & sub[fusion]).sum())
            harmed = int((sub[audio] & ~sub[fusion]).sum())
            gain = 100*(rescued-harmed)/len(sub)
            assert np.isclose(gain, 100*(sub[fusion].mean()-sub[audio].mean()))
            rows.append(dict(grouping=grouping, quintile=q, model=model, N=len(sub),
                             audio_accuracy_pct=100*sub[audio].mean(),
                             fusion_accuracy_pct=100*sub[fusion].mean(),
                             rescued=rescued, harmed=harmed, fusion_gain_pp=gain))
results = pd.DataFrame(rows)
print("\nAccuracy, rescued/harmed, and fusion gains for both grouping schemes:")
print(results.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Verify the original CNN-group results against paper precision, independent of the MM grouping.
references = {"CNN": [17.45,12.32,10.64,-1.96,-19.72],
              "MM": [17.45,12.89,8.40,-3.36,-16.94]}
for model, expected in references.items():
    got = results.query("grouping == 'CNN visual typicality' and model == @model").sort_values("quintile")
    assert len(got)==5 and np.max(np.abs(got.fusion_gain_pp.to_numpy()-expected)) < .011, model
print("\nOriginal CNN-defined baseline reproduced to paper precision: YES")

## 5. Paired artist-cluster bootstrap for the new and original grouping schemes

Artists are resampled with replacement (`B=3000`), retaining all recordings belonging to each selected artist. **Both grouping schemes and both classifiers use the exact same resampled artists within a replicate.** Original quintile memberships remain fixed, so these confidence intervals are **conditional on the retrospective groups**, not independent validation.

In [ ]:
B = 3000
rng = np.random.default_rng(2026)
unique_artists, artist_codes = np.unique(paired.msd_artist_id.astype(str).to_numpy(), return_inverse=True)
A = len(unique_artists)
assert A == 975
schemes = [("CNN visual typicality", "cnn_visual_quintile"),
           ("MM visual typicality", "mm_visual_quintile")]
model_names = ["CNN", "MM"]
counts = np.zeros((2,A,5), dtype=np.int32)
sums = np.zeros((2,A,5,2), dtype=np.float64)
gains = np.stack([
    paired[fusion].astype(int).to_numpy() - paired[audio].astype(int).to_numpy()
    for _, audio, fusion in comparisons
], axis=1)
for i in range(len(paired)):
    for sc, (_, qcol) in enumerate(schemes):
        a, q = artist_codes[i], int(paired[qcol].iat[i])-1
        counts[sc,a,q] += 1
        sums[sc,a,q,:] += gains[i,:]

boot = np.full((B,2,5,2),np.nan,dtype=float)
for b in range(B):
    selected = rng.integers(0,A,size=A)
    weights = np.bincount(selected,minlength=A)
    cnt = np.einsum('a,saq->sq',weights,counts)
    total = np.einsum('a,saqr->sqr',weights,sums)
    with np.errstate(divide='ignore',invalid='ignore'):
        boot[b] = 100*np.divide(total,cnt[:,:,None],where=cnt[:,:,None]>0,
                                out=np.full((2,5,2),np.nan))

ci_rows=[]
for sc,(name,_) in enumerate(schemes):
    for m,model in enumerate(model_names):
        for q in range(5):
            vals = boot[:,sc,q,m]
            vals = vals[np.isfinite(vals)]
            point = results.query('grouping == @name and model == @model and quintile == @q_plus_one',
                                  local_dict={'name':name,'model':model,'q_plus_one':q+1}).fusion_gain_pp.iloc[0]
            ci_rows.append(dict(grouping=name,model=model,quintile=q+1,
                                fusion_gain_pp=point,
                                ci_lower_pp=np.percentile(vals,2.5),
                                ci_upper_pp=np.percentile(vals,97.5),
                                valid_bootstraps=len(vals)))
ci_table=pd.DataFrame(ci_rows)
assert (ci_table.valid_bootstraps == B).all(), 'Missing quintile bootstrap samples'
print(ci_table.to_string(index=False,float_format=lambda x:f'{x:.2f}'))

# Compare the strength of the Q1−Q5 gradient between the two schemes.
contrast_rows=[]
for m,model in enumerate(model_names):
    original=results.query('grouping == "CNN visual typicality" and model == @model')
    alternative=results.query('grouping == "MM visual typicality" and model == @model')
    c1=(original.loc[original.quintile.eq(1),'fusion_gain_pp'].iat[0]
        -original.loc[original.quintile.eq(5),'fusion_gain_pp'].iat[0])
    c2=(alternative.loc[alternative.quintile.eq(1),'fusion_gain_pp'].iat[0]
        -alternative.loc[alternative.quintile.eq(5),'fusion_gain_pp'].iat[0])
    b1=boot[:,0,0,m]-boot[:,0,4,m]
    b2=boot[:,1,0,m]-boot[:,1,4,m]
    delta=b2-b1
    contrast_rows.append(dict(model=model,
        cnn_group_contrast_pp=c1,mm_group_contrast_pp=c2,
        mm_minus_cnn_contrast_pp=c2-c1,
        mm_group_ci_low_pp=np.percentile(b2,2.5),
        mm_group_ci_high_pp=np.percentile(b2,97.5),
        difference_ci_low_pp=np.percentile(delta,2.5),
        difference_ci_high_pp=np.percentile(delta,97.5)))
contrasts=pd.DataFrame(contrast_rows)
print('\nQ1−Q5 contrast sensitivity to the visual representation:')
print(contrasts.to_string(index=False,float_format=lambda x:f'{x:.2f}'))

## 6. Visualize and export the comparison

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
paired.to_csv(OUTPUT / 'mm_typicality_track_assignments_and_predictions.csv',index=False)
results.to_csv(OUTPUT / 'mm_typicality_fusion_gains.csv',index=False)
ci_table.to_csv(OUTPUT / 'mm_typicality_cluster_bootstrap_ci.csv',index=False)
contrasts.to_csv(OUTPUT / 'mm_typicality_q1_q5_contrasts.csv',index=False)
assignments.to_csv(OUTPUT / 'cnn_vs_mm_quintile_membership.csv')
np.save(OUTPUT / 'mm_typicality_bootstrap_samples.npy',boot)

for model in ['CNN','MM']:
    fig,ax=plt.subplots(figsize=(8,4.8))
    for group in ['CNN visual typicality','MM visual typicality']:
        part=ci_table.query('grouping == @group and model == @model').sort_values('quintile')
        y=part.fusion_gain_pp.to_numpy()
        err=np.vstack([y-part.ci_lower_pp.to_numpy(),part.ci_upper_pp.to_numpy()-y])
        ax.errorbar(part.quintile.to_numpy(),y,yerr=err,fmt='o-',capsize=3,label=group)
    ax.axhline(0,linestyle='--',linewidth=1)
    ax.set_xticks(range(1,6),[f'Q{i}' for i in range(1,6)])
    ax.set_xlabel('Visual typicality quintile (true-genre, retrospective)')
    ax.set_ylabel('Early-fusion accuracy gain (percentage points)')
    ax.set_title(f'{model}: effect of changing the typicality representation')
    ax.legend()
    ax.grid(axis='y',alpha=.2)
    fig.tight_layout()
    fig.savefig(OUTPUT / f'{model.lower()}_cnn_vs_mm_typicality_gain.png',dpi=180,bbox_inches='tight')
    plt.show()
    plt.close(fig)
print('\nSaved all results and figures to:',OUTPUT)

## 7. Interpretation guide — decide *after* viewing the results
The central question is whether the **sign and ordering** of early-fusion gains across Q1–Q5 persist when defining quintiles with MM visual embeddings.

- **Similar progression for CNN and MM:** Supports *robustness to this alternative representation choice*, **not** causal claims or independent generalization.
- **Strongly different progression:** Shows the pattern depends on the representation used to define true-genre typicality; report that difference transparently.
- **Mixed progression:** State that the association is partly representation-dependent; do not cherry-pick only the favorable model or quintiles.
- The table `mm_typicality_q1_q5_contrasts.csv` reports the contrast and its paired bootstrap difference between grouping schemes; these CIs are conditional on fixed group assignments and post-hoc analysis choices.

**Do not retrain** audio or fusion models to force a desired pattern, and do not interpret a high test-label-dependent typicality score as something available to a real inference-time system.

**Suggested next step:** send the executed `.ipynb` or the two exported tables (`mm_typicality_fusion_gains.csv` and `mm_typicality_q1_q5_contrasts.csv`) for a strict scientific interpretation and a short addition to the report (only if supported by results).